In [ ]:
%matplotlib inline
import os
import sys
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision.transforms import Compose
from tqdm import tqdm
import mlflow.pytorch
import imgaug.augmenters as iaa
from pprint import pprint
import pandas as pd
import cv2
import plotly.express as px
import mlflow
import plotly.figure_factory as ff


import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append('../../../')

from src import MODELS_DIR, MLFLOW_TRACKING_URI, DATA_PATH
from src.data import TrainValTestSplitter, MURASubset
from src.data.transforms import *
from src.features.augmentation import Augmentation
from src.models.autoencoders import BottleneckAutoencoder, BaselineAutoencoder
from src.models.gans import DCGAN
from src.models.vaetorch import VAE
from src.models import BaselineAutoencoder
from src.features.pixelwise_loss import PixelwiseLoss
from src.models.autoencoders import MaskedMSELoss
from src.features.topk import TopK
from src.visualization.plot_loss_label import PlotLossLabel

## Best masked model

### Initilize and load model

In [ ]:
# Connect to mlflow
client = mlflow.tracking.MlflowClient(MLFLOW_TRACKING_URI)
client.list_experiments()
# get the path of the saved model from mlflow
run_id = '5ca7f67c33674926a00590752c877fe5'
experiment = client.get_experiment('1')
path = f'{experiment.artifact_location}/{run_id}/artifacts/BaselineAutoencoder.pth'
path

num_workers = 7
log_to_mlflow = False
device = "cpu"

# Mlflow parameters
run_params = {
    'batch_size': 32,
    'image_resolution': (512, 512),
    'num_epochs': 1000,
    'batch_normalisation': True,
    'pipeline': {
        'hist_equalisation': True,
        'data_source': 'XR_HAND_PHOTOSHOP',
    },
    'masked_loss_on_val': True,
    'masked_loss_on_train': True,
    'soft_labels': True,
    'glr': 0.001,
    'dlr': 0.00005,
    'z_dim': 1000,
    'lr': 0.0001
}


# Preprocessing pipeline

composed_transforms_val = Compose([GrayScale(),
                                   HistEqualisation(active=run_params['pipeline']['hist_equalisation']),
                                   Resize(run_params['image_resolution'], keep_aspect_ratio=True),
                                   Augmentation(iaa.Sequential([iaa.PadToFixedSize(512, 512, position='center')])),
                                   # Padding(max_shape=run_params['image_resolution']),
                                   # max_shape - max size of image after augmentation
                                   MinMaxNormalization(),
                                   ToTensor()])

# get data

data_path = f'{DATA_PATH}/{run_params["pipeline"]["data_source"]}'
splitter = TrainValTestSplitter(path_to_data=data_path)

test = MURASubset(filenames=splitter.data_test.path, true_labels=splitter.data_test.label,
                  patients=splitter.data_test.patient, transform=composed_transforms_val)

test_loader = DataLoader(test, batch_size=run_params['batch_size'], shuffle=True, num_workers=num_workers)

# get model (change path to path to a trained model

model = torch.load(path, map_location=lambda storage, loc: storage)

# set loss function

outer_loss = MaskedMSELoss(reduction='none')
model.eval().to(device)

In [ ]:
plotloss = PlotLossLabel(model=model, data=test_loader, device=device, outer_loss=outer_loss,
                         masked_loss_on_val=True, bin_size=0.0002)

In [ ]:
plotloss.evaluation_cae()

In [ ]:
plt = plotloss.get_plot()

In [ ]:
plt.show()

## Best not masked model

### Initilize and load model

In [ ]:
# Connect to mlflow
client = mlflow.tracking.MlflowClient(MLFLOW_TRACKING_URI)
client.list_experiments()
# get the path of the saved model from mlflow
run_id = '0ca304cc755f4203a5c6576edb14e4b9'
experiment = client.get_experiment('1')
path = f'{experiment.artifact_location}/{run_id}/artifacts/baseline_autoencoder/data/model.pth'
path

num_workers = 7
log_to_mlflow = False
device = "cpu"

# Mlflow parameters
run_params = {
    'batch_size': 32,
    'image_resolution': (512, 512),
    'num_epochs': 1000,
    'batch_normalisation': True,
    'pipeline': {
        'hist_equalisation': True,
        'data_source': 'XR_HAND_CROPPED',
    },
    'masked_loss_on_val': True,
    'masked_loss_on_train': True,
    'soft_labels': True,
    'glr': 0.001,
    'dlr': 0.00005,
    'z_dim': 1000,
    'lr': 0.0001
}


# Preprocessing pipeline

composed_transforms_val = Compose([GrayScale(),
                                   HistEqualisation(active=run_params['pipeline']['hist_equalisation']),
                                   Resize(run_params['image_resolution'], keep_aspect_ratio=True),
                                   Augmentation(iaa.Sequential([iaa.PadToFixedSize(512, 512, position='center')])),
                                   # Padding(max_shape=run_params['image_resolution']),
                                   # max_shape - max size of image after augmentation
                                   MinMaxNormalization(),
                                   ToTensor()])

# get data

data_path = f'{DATA_PATH}/{run_params["pipeline"]["data_source"]}'
splitter = TrainValTestSplitter(path_to_data=data_path)

test = MURASubset(filenames=splitter.data_test.path, true_labels=splitter.data_test.label,
                  patients=splitter.data_test.patient, transform=composed_transforms_val)

test_loader = DataLoader(test, batch_size=run_params['batch_size'], shuffle=True, num_workers=num_workers)

# get model (change path to path to a trained model

model = torch.load(path, map_location=lambda storage, loc: storage)

# set loss function

outer_loss = MaskedMSELoss(reduction='none')
model.eval().to(device)

In [ ]:
plotloss = PlotLossLabel(model=model, data=test_loader, device=device, outer_loss=outer_loss,
                         masked_loss_on_val=True, bin_size=0.0002)

plotloss.evaluation_cae()

plt = plotloss.get_plot()

plt.show()